# Building an MLP for weigths recovery in Pott's Model
This file defined an MLP that will try to learn the weights in profile and Pott's model, this consideration is due to a difficulty in finding a good penalization factor during ridge regression, the too many weights create a situation where the best lambda is extremely high, and therefore no choices are made and the predicted weights are too near the mean, therefore zero        

### Import of useful libraries

In [1]:
!pip install -U "jax[cuda12]"
!pip install -U flax

INFO: pip is looking at multiple versions of jax[cuda12] to determine which version is compatible with other requirements. This could take a while.
  Using cached jax-0.11.0-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.10.2-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.10.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (1.3 kB)
  Using cached jax-0.10.1-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.10.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (1.3 kB)
  Using cached jax-0.10.0-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.10.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (1.3 kB)
INFO: pip is still looking at multiple versions of jax[cuda12] to determine which version is compatible with other requirements. This could take a while.
  Using cached jax-0.9.2-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.9.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (1.3 kB)
  Using cached jax-0.9.1-py3-none-any.whl.metadata (13 kB)
  Using cached ja

In [15]:
import numpy as np
import matplotlib.pyplot as plt
import jax 
import jax.numpy as jnp

from flax import nnx
from functools import partial
from typing import Optional
import optax

### First simple MLP

In [ ]:
class SimpleMLP(nnx.Module):
    """ 
    Simple MLP
    """
    
    def __init__(self, input_dim:int, hidden_dim:int, output_dim:int, *, rngs: nnx.Rngs):
        self.linear1    = nnx.Linear(input_dim, hidden_dim, rngs=rngs)
        self.dropout    = nnx.Dropout(rate=0.1, rngs=rngs)
        self.batchnorm  = nnx.BatchNorm(hidden_dim, use_running_average=False, rngs=rngs)
        self.linear2    = nnx.Linear(hidden_dim, output_dim, rngs=rngs)
        
    def __call__(self, x:jax.Array, rngs: nnx.Rngs):
        x = nnx.gelu(self.dropout(self.batchnorm(self.linear1(x)), rngs=rngs))
        return self.linear2(x)

model = SimpleMLP(2, 256, 16, rngs=nnx.Rngs(0))

y = model(x=jnp.ones((3,2)), rngs=nnx.Rngs(1))

nnx.display(model)

### Adding an optimizer and train step

In [ ]:
model = SimpleMLP(2, 16, 10, rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=0.05), wrt=nnx.Param)

@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    """ 
    x : 7 AA sequence
    y : 140 profile weights of the sequence
    """
    def loss_fn(model: SimpleMLP, rngs: nnx.Rngs):
        y_pred = model(x,rngs)
        return jnp.mean((y_pred-y)**2)
    
    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
     
    return loss

x, y = jnp.ones((5, 2)), jnp.ones((5, 10))
loss = train_step(model, optimizer, x, y, rngs)

print(f'{loss = }')
print(f'{optimizer.step.value = }')

loss = Array(1.0000052, dtype=float32)
optimizer.step.value = Array(1, dtype=uint32)


/var/folders/xf/g2dzx3814xx2yfk3fln8r_zc0000gn/T/ipykernel_12039/489737546.py:20: DeprecationWarning: '.value' access is now deprecated. For Variable[Array] instances use:

  variable[...]

For other Variable types use:

  variable.get_value()

  print(f'{optimizer.step.value = }')


In [28]:
@nnx.jit
def train_epoch(model, optimizer, X, Y, rngs):
    """ 
    X shape(N, 7)     : batch of N sequences of 7 AA 
    Y shape(N, 140)   : profile weights of the N sequences 
    """
    for i in range(N):
        loss = train_step(model, optimizer, X[i], Y[i])
        if i % 10 == 0:
            print(f"Setp {optimizer.step.value}, loss: {loss}")
    

### Auto-adjustable lr for optimizer

In [27]:
total_training_steps   = 1000
warmup_fraction     = 0.1
peak_lr             = 1e-3
final_lr            = 1e-5

lr_schedule_fn = optax.warmup_cosine_decay_schedule(
    init_value=0.,
    peak_value=peak_lr,
    warmup_steps=int(total_training_steps * warmup_fraction),
    decay_steps=int(total_training_steps * (1. - warmup_fraction)),
    end_value=final_lr
)

optimizer_scheduled_lr = nnx.Optimizer(model, optax.adamw(learning_rate=lr_schedule_fn), wrt=nnx.Param)

# Adapted to profile prediction

The entry is a sequence of 7 amino-acids and the output is the 140 profile weights

In [29]:
Profile_model = SimpleMLP(7, 256, 140, rngs=nnx.Rngs(42))

nnx.display(Profile_model)